## Set Price regime

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

<div dir="rtl" align="right">
خواندن خروجی مرحله ی قبل
</div>

In [ ]:
df = pd.read_feather("../Outputs/01_df.feather")

<div dir="rtl" align="right">
تنظیمات نمایش داده
</div>

In [ ]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

<div dir="rtl" align="right">
حذف آگهی‌های نامعتبر

اگهی های فروش که فیلد رهن و اجاره را پر کرده اند یا بالعکس داده های نامعتبر هستند. 
</div>

In [ ]:
is_rent = df['cat2_slug'].isin(['commercial-rent', 'residential-rent'])
is_sell = df['cat2_slug'].isin(['commercial-sell', 'residential-sell'])

drop_sale = is_sell & (df["rent_value"].notna() | df["credit_value"].notna() | df["rent_mode"].notna() | df["credit_mode"].notna())
drop_rent = is_rent & (df["price_value"].notna() | df["price_mode"].notna())


df = df.drop(df[drop_sale | drop_rent].index)

<div dir="rtl" align="right" >
 رژیم قیمتی
</div>

In [ ]:

def define_price_regime(df):
    df['price_regime'] = 'unknown'
    df['price_status'] = 'unknown'

    df['sale_price'] = np.nan
    df['monthly_rent'] = np.nan
    df['deposit_amount'] = np.nan

    # --- Rental rules ---
    is_rent = df['cat2_slug'].isin(['commercial-rent', 'residential-rent'])
    is_sell = df['cat2_slug'].isin(['commercial-sell', 'residential-sell'])

    
    conds_rent = [
        is_rent & (df['rent_mode'] == 'مقطوع') & (df['credit_mode'] == 'مقطوع'),
        is_rent & (df['rent_mode'] == 'مقطوع') & (df['credit_mode'] == 'توافقی') & (df['rent_value'] > 0),
        is_rent & (df['rent_mode'] == 'مقطوع') & (df['credit_mode'] == 'مجانی'),
        is_rent & (df['rent_mode'] == 'توافقی') & (df['credit_mode'] == 'مقطوع'),
        is_rent & (df['rent_mode'] == 'توافقی') & (df['credit_mode'] == 'توافقی'),
        is_rent & (df['rent_mode'] == 'توافقی') & (df['credit_mode'] == 'مجانی'),
        is_rent & (df['rent_mode'] == 'مجانی') & (df['credit_mode'] == 'مقطوع'),
        is_rent & (df['rent_mode'] == 'مجانی') & (df['credit_mode'] == 'توافقی'),
        is_rent & (df['rent_mode'] == 'مجانی') & (df['credit_mode'] == 'مجانی'),
        is_rent & (df['rent_mode'].isna()) & (df['credit_mode'].isna()),
    ]
    
    # English replacements:
    regimes_rent = [
        'mortgage_and_rent',  # رهن و اجاره
        'mortgage_and_rent',
        'rent_only',          # فقط اجاره
        'mortgage_and_rent',
        'mortgage_and_rent',
        'rent_only',
        'mortgage_only',      # فقط رهن
        'mortgage_only',
        'unknown',
        "mortgage_and_rent"
    ]
    
    statuses_rent = [
        'valid',              # معتبر
        'negotiable',         # توافقی
        'valid',
        'negotiable',
        'negotiable',
        'negotiable',
        'valid',
        'negotiable',
        'Invalid',
        'NoInformation'
    ]
    
    rent_regime_arr = np.select(conds_rent, regimes_rent, default='unknown')
    rent_status_arr = np.select(conds_rent, statuses_rent, default='unknown')
    
    df.loc[is_rent, 'price_regime'] = rent_regime_arr[is_rent]
    df.loc[is_rent, 'price_status'] = rent_status_arr[is_rent]

    # --- Sales rules ---
    
    sell_valid = is_sell & (df['price_value'] > 0) & (df['price_mode'] == 'مقطوع')
    sell_nego  = is_sell & (df['price_mode'] == 'توافقی')
    sell_free  = is_sell & (df['price_mode'] == 'مجانی')
    sell_noInfo  = is_sell & (df['price_mode'].isna())
    
    df.loc[sell_valid, 'price_regime'] = 'sell'          # فروش
    df.loc[sell_valid, 'price_status'] = 'valid'         # معتبر
    df.loc[sell_nego, 'price_regime']  = 'sell'
    df.loc[sell_nego, 'price_status']  = 'negotiable'   # توافقی
    df.loc[sell_free, 'price_regime']  = 'sell'
    df.loc[sell_free, 'price_status']  = 'inconsistent' # ناسازگار
    df.loc[sell_noInfo, 'price_regime']  = 'sell'
    df.loc[sell_noInfo, 'price_status']  = 'missed' # مفقود


    #------------- Add valid Value Column------------------
    valid_mask = df['price_status'] == 'valid'

    has_size = df['building_size'].notna() & (df['building_size'] > 0)

    # ۱. فروش معتبر
    mask_sell = valid_mask & (df['price_regime'] == 'sell')
    df.loc[mask_sell, 'sale_price'] = df.loc[mask_sell, 'price_value']

    # ۲. رهن و اجاره معتبر
    mask_mortgage_rent = valid_mask & (df['price_regime'] == 'mortgage_and_rent')
    df.loc[mask_mortgage_rent, 'monthly_rent'] = df.loc[mask_mortgage_rent, 'rent_value']
    df.loc[mask_mortgage_rent, 'deposit_amount'] = df.loc[mask_mortgage_rent, 'credit_value']

    # ۳. فقط اجاره معتبر
    mask_rent_only = valid_mask & (df['price_regime'] == 'rent_only')
    df.loc[mask_rent_only, 'monthly_rent'] = df.loc[mask_rent_only, 'rent_value']
    df.loc[mask_rent_only, 'deposit_amount'] = 0   # بدون رهن

    # ۴. فقط رهن معتبر
    mask_mortgage_only = valid_mask & (df['price_regime'] == 'mortgage_only')
    df.loc[mask_mortgage_only, 'deposit_amount'] = df.loc[mask_mortgage_only, 'credit_value']
    df.loc[mask_mortgage_only, 'monthly_rent'] = 0   # بدون اجاره

    
    return df

<div dir="rtl" align="right">
اعمال رژیم قیمتی روی داده‌ها و چاپ خلاصه تعداد آگهی‌ها بر اساس دسته‌بندی و وضعیت قیمت
</div>

In [ ]:
df = define_price_regime(df)

# جدول ترکیبی
print(df.groupby(['cat2_slug','price_regime', 'price_status']).size())

In [ ]:
invalid_sell= df[ (df["price_status"]=="Invalid")]

invalid_sell[
    [
        "cat2_slug",
        "title",
        "price_mode",
        "price_value",
        "sale_price",
        "rent_mode",
        "credit_mode",
        "rent_value",
        "credit_value",
        "description"
    ]
].head(20)

Save Output

In [ ]:
df.to_feather("../Outputs/02_df.feather")
